In [14]:
from youtube_transcript_api import YouTubeTranscriptApi
from urllib.parse import urlparse, parse_qs

def parse_url(url: str) -> str:
    """
    Extract video ID from URL.

    Args:
        url(str): Youtube video url
    
    Returns:
        Youtube video's video ID
    """

    parsed_url = urlparse(url)
    query_params = parse_qs(parsed_url.query)

    print("Query Parameters : ", query_params)

    if "v" in query_params:
        return query_params["v"][0]

    return "no_video_id"

parse_url("https://www.youtube.com/watch?v=NMyg91iVZdE&t=11s")

Query Parameters :  {'v': ['NMyg91iVZdE'], 't': ['11s']}


'NMyg91iVZdE'

In [13]:
def get_text_from_video(url: str) -> str:
    """
    Get transcript text from YouTube video.

    Args:
        url(str): youtube video url

    Returns:
        Youtube video's transcripted text
    
    """

    video_id = parse_url(url)

    try:
        transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=['te'])
        print(f"Transcirpt : {transcript}")
        transcript_text = " ".join([entry["text"] for entry in transcript])
        transcript_text = transcript_text.replace("\n", " ").replace("'", "")
        return transcript_text
    except Exception as e:
        return f"Failed to retrieve transcript: {str(e)}"

video_url = "https://www.youtube.com/watch?v=wQhf4ccRuUQ"
transcript = get_text_from_video(video_url)
transcript


✅ Extracted video ID: wQhf4ccRuUQ


'Failed to retrieve transcript: no element found: line 1, column 0'

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def create_chunks(transcript_text: str) -> list:
    """
    Split transcript text into processable chunks.

    Args:
        transcript_text (str): Youtube video's transcripted text

    Returns:
        processable chunks
    
    """

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=500)
    chunks = text_splitter.split_text(transcript_text)
    return chunks


chunks = create_chunks(transcript)
chunks


['[Music] hello everyone I hope you are doing extremely well in this video we are going to develop a generative AI application this application helps the students in their exam preparation here we implement the rag agent the rag is nothing but retrieval argumented generation like well give uh content or PDFs to the rag agent or chatbot then chatbot used to generate the answers based on the content which we provided lets see the demo of this project a student can select any model from here deeps llama llama 70p Gemini and mol and can test any model here Im using Lama IDP and the first question I asked here hello Im sto who are you the model generated a nice uh response and as well introduced about this agent and I asked provide me the important questions from entrepreneurship and new ventur it given all the important questions from like different categories entrepreneural education planed publicity for entrepreneur opportunities and many more and asked again what are the benefits and li

In [23]:
from langchain_core.prompts import ChatPromptTemplate

System_Prompt = ChatPromptTemplate.from_template(
        """
        You are an expert YouTube Transcript Summarizer. Your task is to analyze the transcript of a YouTube video and generate a concise, well-structured summary that helps users quickly understand the content. Follow these guidelines:

        1. **Structure the Summary**:
        - Start with a brief overview of the video’s topic.
        - Highlight the main points or sections covered in the video.
        - End with a concise conclusion or key takeaway.

        2. **Focus on Clarity**:
        - Use simple and easy-to-understand language.
        - Avoid unnecessary details and focus on the most relevant information.

        3. **Organize Information**:
        - Use bullet points or numbered lists for clarity where applicable.
        - Group similar ideas together to streamline understanding.

        4. **Adapt to the Content**:
        - For educational videos: Emphasize key concepts and actionable insights.
        - For tutorials: Highlight step-by-step instructions or processes.
        - For interviews: Summarize key questions and responses.

        5. **Formatting**:
        - Use headings (e.g., Introduction, Key Points, Conclusion) to organize the summary.
        - Highlight important terms or phrases for emphasis.
        
        <context>
        {context}
        </context>

        Question: {input}
        """
    )

In [24]:
import re

def remove_think_tags(text):
    pattern = r'<think>.*?</think>'
    cleaned_text = re.sub(pattern, '', text, flags=re.DOTALL)
    return cleaned_text.strip()


In [25]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.chains.combine_documents import create_stuff_documents_chain

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

def get_final_summary_from_chunk_summaries(chunks: list) -> str:
    """
    Summarize text chunks and create a single summary.
    
    Args:
        chunks (list): processable chunks of transcriptted text

    Returns:
        A single summary for youtube video
    """

    llm = ChatGroq(groq_api_key=groq_api_key, model_name="llama3-8b-8192")

    qa_chain = create_stuff_documents_chain(llm, System_Prompt)

    summaries = [qa_chain.invoke(
        {
            "input": chunk,
            "context": "",
        }) for chunk in chunks]

    
    print(f"Summaries :  {summaries}\n")

    combined_summary = [remove_think_tags(summary) for summary in summaries]
    combined_summary = " ".join(combined_summary)


    final_summary = qa_chain.invoke({
        "input": combined_summary,
        "context": "",
    })

    return final_summary


summary = get_final_summary_from_chunk_summaries(chunks)

Summaries :  ["**Summary:**\n\nIn this video, the creator develops a generative AI application to aid students in their exam preparation. The application employs a Retrieval-Augmented Generation (RAG) agent, which takes in content or PDFs and generates answers based on the provided material. The creator demonstrates the project, showcasing the RAG agent's capabilities, such as generating responses to questions and remembering user history.\n\n**Key Points:**\n\n* The RAG agent takes in user queries and generates responses based on pre-trained data.\n* The agent uses a rag module to pitch related content to the user query, overcoming limitations such as hallucination.\n* The creator demonstrates the RAG agent's capabilities, including:\n\t+ Generating responses to questions on entrepreneurship and new ventures.\n\t+ Remembering user history and generating responses accordingly.\n\t+ Providing important questions from different categories.\n\n**Conclusion:**\n\nThe RAG agent has the pote

In [26]:
print(f"Final Summary : {summary}")

Final Summary : Here is a concise and well-structured summary of the YouTube video:

**Introduction**

The video discusses the implementation of a generative AI application to aid students in their exam preparation. The application employs a Retrieval-Augmented Generation (RAG) agent, which takes in content or PDFs and generates answers based on the provided material.

**Key Points**

* The RAG agent uses a rag module to retrieve related content from a knowledge base and generate responses to user queries.
* The agent can remember user history and generate responses accordingly.
* The video covers the practical implementation of a RAG Agent project, including setting up the environment, importing required libraries, and creating a document using the LLM model.

**Conclusion**

The RAG agent has the potential to revolutionize exam preparation by providing students with personalized and accurate responses to their questions. By understanding the capabilities and limitations of this techn

In [30]:

def get_summary_for_video_url(video_url: str) -> str:

    transcript = get_text_from_video(video_url)

    print("Transcript : ", transcript)

    chunks = create_chunks(transcript)

    print("Chunks : ", chunks)

    summary = get_final_summary_from_chunk_summaries(chunks)

    return summary


video_url = "https://www.youtube.com/watch?v=_xol-kbTTFs"    
summary = get_summary_for_video_url(video_url)

Query Parameters :  {'v': ['_xol-kbTTFs']}
Transcirpt : [{'text': 'hello everyone I hope you are doing', 'start': 2.0, 'duration': 4.759}, {'text': 'extremely well today we are going to', 'start': 3.959, 'duration': 5.72}, {'text': 'develop a project that is all-in-one', 'start': 6.759, 'duration': 5.8}, {'text': 'chart application so here we are going', 'start': 9.679, 'duration': 6.321}, {'text': 'to use all open source models like the', 'start': 12.559, 'duration': 6.121}, {'text': 'recently released deeps R1 model llama', 'start': 16.0, 'duration': 7.56}, {'text': '8p llama 70p Gemini and mistol and we', 'start': 18.68, 'duration': 7.04}, {'text': 'also use the parameters like temperature', 'start': 23.56, 'duration': 5.6}, {'text': 'max tokens topy and frequency penalty to', 'start': 25.72, 'duration': 5.76}, {'text': 'generate a response based on on our', 'start': 29.16, 'duration': 4.84}, {'text': 'requirement and also see these', 'start': 31.48, 'duration': 5.16}, {'text': 'par

In [31]:
print(remove_think_tags(summary))

**Summary**

The video series focuses on developing an all-in-one chart application using open-source large language models (LLMs). The project aims to generate responses based on user requirements and explore how different parameters affect the generated responses.

**Main Points**

* Creating a virtual environment for the project
* Setting up a Python project and installing required modules and libraries
* Configuring the LLM model and creating a simple chart app
* Integrating the LLM with a user interface (UI) to generate responses to user queries

**Conclusion**

By following the steps outlined in this video series, developers can create a functional chatbot with a user-friendly interface. The speakers demonstrate how to set up the project, configure the LLM model, and integrate it with a UI to generate responses. This project showcases the capabilities of open-source LLMs and provides a valuable resource for developers working on similar projects.

**Key Takeaways**

* Open-source

In [ ]:
input_text = """
<text start="0.32" dur="7.519">డే డే టేస్ట్ డైలీ టేస్ట్ డైలీ హాలిడేస్</text>
<text start="4.799" dur="4.561">నాలే అది జిటి హాలిడేస్ తా సౌత్ ఇండియాస్</text>
<text start="7.839" dur="3.281">నెంబర్ వన్ ట్రావెల్ బ్రాండ్ కొన్ని</text>
<text start="9.36" dur="4.48">రోజులు అనొచ్చో చాలా రోజులు అనొచ్చో నాకు</text>
<text start="11.12" dur="5.04">తెలియదు కానీ ఆ ఆఫ్టర్ ఏ వైల్ బాగా</text>
<text start="13.84" dur="5.359">సాటిస్ఫయింగ్ అనిపించిన మూవీ ఈ కోర్ట్ టు</text>
<text start="16.16" dur="5.84">స్టార్ట్ విత్ శివాజీ గారు ఆయన ఆయన</text>
<text start="19.199" dur="5.121">పర్ఫార్మెన్స్ అసలు ఆయన ఉన్న ప్రతి సీన్</text>
<text start="22" dur="5.599">లో ఆయన స్క్రీన్ ని కమాండ్ చేస్తారు ద వే</text>
<text start="24.32" dur="5.68">హి ఆయన మీసాలు ఇలా ఇలా అనుకునే ఒక మానరిజం</text>
<text start="27.599" dur="4.48">కానివ్వండి లేకపోతే ఆయన కళ్ళతో ఆయన కోపం</text>
<text start="30" dur="4.559">వచ్చిన ప్రతిసారి ఆయనకి చిరాకు వచ్చిన</text>
<text start="32.079" dur="5.281">ప్రతిసారి కళ్ళు ఒకలాగా పెడతారు ఆయన అసలు</text>
<text start="34.559" dur="6.961">ఆ మానరిజమ్స్ ఆ బాడీ లాంగ్వేజ్ ఆయన ఒకరి</text>
<text start="37.36" dur="7.6">మీద కోపం గాని వచ్చిందంటే ఒక ఒక రూడ్ గా</text>
<text start="41.52" dur="6.08">అరుస్తారు అది ప్రతిదీ ఇట్స్ జస్ట్ టూ</text>
<text start="44.96" dur="4.4">గుడ్ ఒక టెంప్లేటిష్ విలన్ లాంటి రోల్</text>
<text start="47.6" dur="4">అంటే అలాంటి రోల్స్ మనం చాలా సార్లు</text>
<text start="49.36" dur="5.12">చూస్తున్నాం ఒక ఒక క్లాస్ అబ్సెస్డ్ ఒక</text>
<text start="51.6" dur="5.119">కాస్ట్ అబ్సెస్డ్ టిపికల్ ఆ ఒక మేల్</text>
<text start="54.48" dur="4">చావనిస్ట్ అటువంటి అంత టెంప్లేట్</text>
<text start="56.719" dur="3.281">క్యారెక్టర్ ని కూడా శివాజీ గారు ఆ</text>
<text start="58.48" dur="4.16">కాస్టింగ్ వల్ల చాలా ఫ్రెష్ గా</text>
<text start="60" dur="4.799">అనిపించింది ఆయనకి శుభలేఖ సుధాకర్ గారికి</text>
<text start="62.64" dur="4.159">సెకండ్ హాఫ్ లో ఒక చిన్న ఇంటరాక్షన్ ఉంటది</text>
<text start="64.799" dur="3.841">అంటే ఆయన స్క్రీన్ మీదకి వచ్చిన ప్రతిసారి</text>
<text start="66.799" dur="3.68">విల్ బి లైక్ అబ్బా వచ్చాడురా అనే ఒక</text>
<text start="68.64" dur="4.64">ఫీలింగ్ లో ఉంటది అన్నమాట ఆయన స్క్రీన్</text>
<text start="70.479" dur="5.361">మీద కనిపిస్తే చాలు యు విల్ ఫీల్ సంథింగ్</text>
<text start="73.28" dur="4.72">లైక్ ఐదర్ అనోయిడ్ ఐదర్ హేట్ ఆర్ ఐదర్ యు</text>
<text start="75.84" dur="4.24">విల్ లాఫ్ హిట్ హిస్ రూడ్నెస్ లైక్ ఐ సెడ్</text>
<text start="78" dur="4.479">వన్ ఆఫ్ ది గ్రేటెస్ట్ పర్ఫార్మెన్సెస్</text>
<text start="80.08" dur="5.359">ఇన్ తెలుగు సినిమా రీసెంట్ టైమ్స్ మేజర్</text>
<text start="82.479" dur="4.881">మేజర్ మేజర్ క్రెడిట్స్ ఫర్ ద కాస్టింగ్</text>
<text start="85.439" dur="5.601">అంటే ఇట్స్ ఏ ప్రాపర్ ఆన్సంబుల్ కాస్ట్</text>
<text start="87.36" dur="5.92">కదా ఎవ్రీబడీ ఎవ్రీ యాక్టర్ ఇస్ లైక్</text>
<text start="91.04" dur="4.64">పర్ఫెక్ట్ ఇన్ దేర్ రోల్ అపోనెంట్ లాయర్</text>
<text start="93.28" dur="4.799">కింద హర్షవర్ధన్ గారు కానివ్వండి రోహిణి</text>
<text start="95.68" dur="5.439">గారు కానివ్వండి ద లీడ్స్ రోషన్ అండ్</text>
<text start="98.079" dur="6.08">శ్రీదేవి వాళ్ళు కూడా ఈ టీనేజ్ లవ్ స్టోరీ</text>
<text start="101.119" dur="5.04">ఏదైతే ఉందో వాళ్ళ పర్ఫార్మెన్స్ దాన్ని</text>
<text start="104.159" dur="4.24">క్రింజ్ ఆ బార్డర్ లైన్ క్రింజ్</text>
<text start="106.159" dur="4">దాటనివ్వకుండా చూసుకుంటాయి అది చాలా థిన్</text>
<text start="108.399" dur="3.521">లైన్ కదా ఎప్పుడన్నా కూడా ఇటు వెళ్తే</text>
<text start="110.159" dur="4">క్రింజ్ అయిపోద్ది ఇటు ఉంటే క్యూట్ ఉంటది</text>
<text start="111.92" dur="4.64">సో హౌ డు యు బ్యాలెన్స్ దట్ అది రైటింగ్</text>
<text start="114.159" dur="5.92">తో పాటు ఈ యాక్టర్స్ ఎవరైతే ఉన్నారో వాళ్ళ</text>
<text start="116.56" dur="5.36">పర్ఫార్మెన్సెస్ వల్ల అది ఒక ఒక ఇబ్బంది</text>
<text start="120.079" dur="3.72">పెట్టే జోన్ లోకి వెళ్ళదు లేకపోతే మనం</text>
<text start="121.92" dur="4.799">చూస్తుంటాం టీనేజర్స్ సినిమాటిక్ గా</text>
<text start="123.799" dur="5.08">మాట్లాడేది ఆ పెద్ద పెద్ద వాక్యాలు</text>
<text start="126.719" dur="4.16">చెప్పేసేది అవి అవి అవి సింక్ అవ్వవు</text>
<text start="128.879" dur="4.241">ఎప్పుడూ కూడా అవి చాలా అతికించినట్టు</text>
<text start="130.879" dur="3.921">ఉంటాయి ఇందులో దే బిహేవ్ లైక్ కిడ్స్</text>
<text start="133.12" dur="3.52">ఓన్లీ దే డోంట్ బిహేవ్ లైక్ అడల్ట్స్ అండ్</text>
<text start="134.8" dur="3.76">మోర్ ఇంపార్టెంట్లీ ప్రియదర్శి ఐ థింక్</text>
<text start="136.64" dur="3.599">దిస్ ఇస్ హిస్ హిస్ బెస్ట్ రోల్ సో ఫార్</text>
<text start="138.56" dur="3.36">అదే చెప్తున్నా కదా దిస్ ఇస్ వన్ ఆఫ్ ది</text>
<text start="140.239" dur="3.921">బెస్ట్ యాక్టెడ్ తెలుగు ఫిల్మ్స్ ఇన్</text>
<text start="141.92" dur="3.84">రీసెంట్ టైమ్స్ ఆల్సో ఇట్ స్ట్రైక్స్ ఏ</text>
<text start="144.16" dur="3.68">వెరీ గుడ్ బాలెన్స్ బిట్వీన్ బీయింగ్ ఏ</text>
<text start="145.76" dur="4.559">లవ్ స్టోరీ అండ్ కోట్ రూమ్ డ్రామా ఇన్</text>
<text start="147.84" dur="4.16">ఫాక్ట్ ఐ ఫౌండ్ ద లవ్ స్టోరీ నియర్లీ</text>
<text start="150.319" dur="3.521">ఫ్లాలెస్ కోట్ రూమ్ డ్రామా తో కొన్ని</text>
<text start="152" dur="3.84">ఇష్యూస్ ఉన్నాయి కొన్ని కొన్ని చోట్ల</text>
<text start="153.84" dur="4.56">ఎస్పెషల్లీ కోట్ రూమ్ డ్రామా సెకండ్ హాఫ్</text>
<text start="155.84" dur="4.32">లో మూవ్ అయ్యేటప్పుడు కొన్ని చోట్ల ఇది</text>
<text start="158.4" dur="3.28">ఇది మరీ సింపుల్ గా ఉందే ఇది కొంచెం</text>
<text start="160.16" dur="3.6">కన్వీనియంట్ గా రాసుకున్నారే అనే ఒక</text>
<text start="161.68" dur="5.44">ఫీలింగ్ అయితే నాకు అనిపించింది కేసులో ఒక</text>
<text start="163.76" dur="5.36">ఒక ప్రాబ్లం ఉంది దాన్ని సూర్యతేజ అనే</text>
<text start="167.12" dur="3.92">క్యారెక్టర్ ప్రియదర్శి ప్లే చేసింది అతను</text>
<text start="169.12" dur="3.839">క్రాక్ చేయాలి అది కొంచెం ఈజీగా</text>
<text start="171.04" dur="3.839">అయిపోయినట్టు అనిపిస్తది అంటే అతను ఎక్కడ</text>
<text start="172.959" dur="3.441">కూడా పెద్ద కష్టపడినట్టు అనిపించదు హి ఇస్</text>
<text start="174.879" dur="3.44">స్మార్ట్ హి ఇస్ డూయింగ్ హిస్ హోం వర్క్</text>
<text start="176.4" dur="4.4">బట్ వి నెవర్ గెట్ టు సీ దట్ లైక్ వైస్</text>
<text start="178.319" dur="5.441">కొన్ని కన్వీనియంట్ రైటింగ్ ఛాయిసెస్ ఆ ఒక</text>
<text start="180.8" dur="6">సపోజ్ ఒక ప్రాబ్లం ఉంది అనుకోండి కేసులో</text>
<text start="183.76" dur="5.92">దాన్ని ఇతను ఎలా దాటుతాడు అనే రకంగా</text>
<text start="186.8" dur="4.719">కాకుండా ఇతను దాటేటట్టు మనం ఒక బ్యాక్</text>
<text start="189.68" dur="4">స్టోరీ పెడదాం ఆల్రెడీ అన్నట్టు రివర్స్</text>
<text start="191.519" dur="4.961">ఇంజనీరింగ్ చేసినట్టు కొన్ని కన్వీనియంట్</text>
<text start="193.68" dur="5.68">గా ప్లేస్ అయి ఉంటాయి సో దాని వల్ల కూడా</text>
<text start="196.48" dur="6.16">మరీ ఈజీగా జరిగిపోతుంది కేసు అనే ఒక</text>
<text start="199.36" dur="5.92">ఫీలింగ్ వస్తది బట్ బట్ బట్ నాకు ఇది</text>
<text start="202.64" dur="4.959">అనిపించినప్పటికీ ఎవ్రీ టైం హి అవుట్</text>
<text start="205.28" dur="4.239">స్మార్ట్స్ హర్షవర్ధన్ హూ ఇస్ హిస్</text>
<text start="207.599" dur="4.481">అపోనెంట్ ఒక రివార్డింగ్ ఫీలింగ్ అని</text>
<text start="209.519" dur="4.321">ఉంటది ఒక ఐ వస్తది ఎందుకంటే ఇట్ ఇస్ ఆల్సో</text>
<text start="212.08" dur="4.159">లైక్ ఏ కమింగ్ ఆఫ్ ఏజ్ టేల్ ఫర్</text>
<text start="213.84" dur="4.56">ప్రియదర్శి సూర్యతేజ అండ్ దానికి</text>
<text start="216.239" dur="4.481">బ్యూటిఫుల్ గా ప్రొనౌన్స్ చేసే సీన్</text>
<text start="218.4" dur="4.16">వచ్చేసి వన్ కాన్వర్సేషన్ హి హాస్ విత్</text>
<text start="220.72" dur="3.92">సాయి కుమార్ అండ్ సాయి కుమార్ గారి</text>
<text start="222.56" dur="4.08">క్యారెక్టర్ కూడా అది అంటే ఇప్పుడు ఒక</text>
<text start="224.64" dur="4.239">టిపికల్ లాయర్ కి ఒక బాస్ కింద ఉన్నాడు</text>
<text start="226.64" dur="4">అనుకోండి దే టెండ్ టు మేక్ హిమ్ ఆల్సో వన్</text>
<text start="228.879" dur="3.92">ఆఫ్ ది యాంటగనిస్ట్ అతన్ని కూడా ఒక</text>
<text start="230.64" dur="3.44">అపోనెంట్ కింద చూపించడం అది ఈజీగా చేసి</text>
<text start="232.799" dur="3.601">ఉండొచ్చు చేసుంటే ఇంకా డ్రామా</text>
<text start="234.08" dur="4.56">పెరుగుండేదేమో బట్ అది చేయకపోవడం చాలా</text>
<text start="236.4" dur="4.32">రిఫ్రెషింగ్ గా ఉంది మూవీ లో ది బెస్ట్</text>
<text start="238.64" dur="4">సీన్ హాస్ టు బి సెకండ్ హాఫ్ లో ఉండే</text>
<text start="240.72" dur="4.799">ప్రియదర్శి అండ్ సాయి కుమార్ వాళ్ళ మధ్య</text>
<text start="242.64" dur="4.959">జరిగే ఒక ఇంటరాక్షన్ లాయర్ అనేవాడు ఒక</text>
<text start="245.519" dur="4">రియల్ లాయర్ అనేవాడు ఏం చేయాలి వాట్</text>
<text start="247.599" dur="3.601">డిఫైన్స్ ఏ లాయర్ అనేదానికి ఆయన ఒక</text>
<text start="249.519" dur="3.521">ఎక్స్ప్లనేషన్ ఇస్తారు బ్యూటిఫుల్ సీన్</text>
<text start="251.2" dur="5.759">అది ఆన్ వన్ లెవెల్ ద ఫిల్మ్ ఇస్ వర్కింగ్</text>
<text start="253.04" dur="5.919">అస్ అస్ ఏ చేంజ్ ఫర్ సూర్యతేజ అండ్ ఒక లవ్</text>
<text start="256.959" dur="3.601">స్టోరీ కింద వర్క్ అవుతది థర్డ్ లెవెల్</text>
<text start="258.959" dur="3.281">ఆబ్వియస్లీ అస్ ఏ కోట్ రూమ్ డ్రామా కింద</text>
<text start="260.56" dur="4.24">కూడా వర్క్ అవుతుంది కాబట్టి ఆ కోర్ట్</text>
<text start="262.24" dur="4.64">రూమ్ డ్రామా లో ఉండే కన్వీనియంట్ రైటింగ్</text>
<text start="264.8" dur="4.24">ని ఐ యామ్ ఏబుల్ టు ఓవర్ లుక్ ద సక్సెస్</text>
<text start="266.88" dur="5.12">ఆఫ్ ఎనీ కోర్ట్ రూమ్ డ్రామా ఇస్ డిఫైన్డ్</text>
<text start="269.04" dur="4.719">బై హౌ ఇన్వెస్టెడ్ ఆర్ వి ఇన్ ద కేస్ సడన్</text>
<text start="272" dur="3.28">గా ఏదైనా ఇష్యూ వస్తే విల్ బి లైక్ ఓ మై</text>
<text start="273.759" dur="3.921">గాడ్ హౌ ఇస్ హి గోయింగ్ టు క్రాక్ ఇట్ ఒక</text>
<text start="275.28" dur="5.04">విన్ వచ్చిన ప్రతిసారి వి ఫీల్ ద హై సో ఆ</text>
<text start="277.68" dur="5.28">రకంగా కోర్ట్ వర్క్స్ నాకు ఆ పర్సనల్ గా ఆ</text>
<text start="280.32" dur="4.96">ఇష్యూస్ అనిపించినప్పటికీ కానీ తను గెలిచే</text>
<text start="282.96" dur="4">మూమెంట్స్ సాటిస్ఫయింగ్ గా ఉండే ఓవరాల్ గా</text>
<text start="285.28" dur="3.44">చెప్పాలి అంటే కోర్ట్ ఇస్ ఏ వెల్ మేడ్</text>
<text start="286.96" dur="3.519">ఫిల్మ్ ఆల్మోస్ట్ అన్ని బాక్సెస్ టిక్</text>
<text start="288.72" dur="3.919">చేసుకుంది అండ్ ఐ థింక్ దట్ ఇస్ వేర్ ది</text>
<text start="290.479" dur="4.241">సక్సెస్ ఆఫ్ ద డైరెక్టర్ డెబ్యూ డైరెక్టర్</text>
<text start="292.639" dur="5.28">రామ్ జగదీష్ లైఫ్ కాస్టింగ్ కానివ్వండి</text>
<text start="294.72" dur="5.28">మ్యూజిక్ సినిమాటోగ్రఫీ ఎవ్రీథింగ్ అన్నీ</text>
<text start="297.919" dur="4.481">కరెక్ట్ గా కుదిరాయి అట్లా ప్రతి సినిమాకి</text>
<text start="300" dur="4.08">కుదరదు డైలాగ్స్ అండి అసలు రైటింగ్ చాలా</text>
<text start="302.4" dur="3.519">రోజుల తర్వాత ఎంత మంచి డైలాగ్స్</text>
<text start="304.08" dur="3.679">వినపడ్డాయో పనికి వెళ్ళడానికి ఆఫీస్ కి</text>
<text start="305.919" dur="5.12">వెళ్ళడానికి తేడా ఉందిరా అని చెప్పడము ఒక</text>
<text start="307.759" dur="6.401">19 ఏళ్ల కుర్రాడి 14 ఏళ్ల జీవితం అండ్</text>
<text start="311.039" dur="4.961">వాళ్ళ కుటుంబం చూసే నరకం వాల్యూ టు లాక్స్</text>
<text start="314.16" dur="4.64">ఆ సాయి కుమార్ గారికి ప్రియదర్శి కి</text>
<text start="316" dur="5.12">మధ్యలో ఉండే కాన్వర్సేషన్ లైక్ లాయర్ అంటే</text>
<text start="318.8" dur="4.16">ఏంటి ఆన్సర్ కాదు క్వశ్చన్ చేయాలి అనేది</text>
<text start="321.12" dur="5.359">అండ్ కొన్ని కొన్ని డైలాగ్స్ ముందుగానే</text>
<text start="322.96" dur="5.76">ఫోర్ షాడో చేస్తాయి అన్నమాట రోషన్ ఒకసారి</text>
<text start="326.479" dur="4.56">ఆ అమ్మాయితో మాట్లాడుతున్నప్పుడు ఈ సంకెల</text>
<text start="328.72" dur="4.56">జీవితం మన వల్ల కాదే అండ్ యు ఆబ్వియస్లీ</text>
<text start="331.039" dur="4.961">నో వాట్ హాపెన్స్ లేటర్ దేర్ ఇస్ ఏ లాట్</text>
<text start="333.28" dur="4.479">ఆఫ్ మీనింగ్ ఇన్ ద డైలాగ్స్ లైక్ ఐ సెడ్ ఐ</text>
<text start="336" dur="4.56">యామ్ ఐ యామ్ హ్యాపీ ఇట్స్ ఏ వెల్</text>
<text start="337.759" dur="4.481">డైరెక్టెడ్ ఫిల్మ్ ఒక మంచి డెబ్యూ ఐ యామ్</text>
<text start="340.56" dur="3.359">ఐ యామ్ హ్యాపీ వి హావ్ ఏ డైరెక్టర్ టు</text>
<text start="342.24" dur="4.48">లుక్ అవుట్ ఫర్ ఐ యామ్ నాట్ మేకింగ్ దిస్</text>
<text start="343.919" dur="5.041">అప్ లాస్ట్ ఇయర్ లక్కీ భాస్కర్ అక్టోబర్</text>
<text start="346.72" dur="4.56">30 ఫస్ట్ అప్ 30th ప్రీమియర్స్ చూసి</text>
<text start="348.96" dur="4.72">బయటికి వచ్చేటప్పుడు ఒక సాటిస్ఫాక్షన్</text>
<text start="351.28" dur="4.88">ఉండింది ఆ సాటిస్ఫాక్షన్ నాకు లాస్ట్ త్రీ</text>
<text start="353.68" dur="4.16">ఫోర్ మంత్స్ లో ఈ సినిమా ఇవ్వలేదు నో డిస్</text>
<text start="356.16" dur="3.2">రెస్పెక్ట్ టు ఎనీ ఫిల్మ్స్ లైక్ మనం</text>
<text start="357.84" dur="3.68">సినిమాలు వచ్చినాయి ఐ యామ్ నాట్ సేయింగ్</text>
<text start="359.36" dur="5.6">దట్ ఆ అన్ని బ్యాడ్ ఫిలిమ్స్ అండి మధ్యలో</text>
<text start="361.52" dur="5.84">మన సినిమాలు వచ్చాయి కానీ ఆ అంటే ప్రతి</text>
<text start="364.96" dur="4.32">దాంట్లో ఈ ఇష్యూ ఉంది ఆ ఇష్యూ ఉంది అనేది</text>
<text start="367.36" dur="5.44">ఒకటి కనిపిస్తూ ఉంటది ఒకటి ఆన్ ఎవ్రీ</text>
<text start="369.28" dur="5.28">లెవెల్ సాటిస్ఫై చేసిన మూవీ లేదు కానీ ఇది</text>
<text start="372.8" dur="3.92">ఎవ్రీ లెవెల్ ఐ యామ్ ఐ యామ్ హ్యాపీ</text>
<text start="374.56" dur="4.479">ఎస్పెషల్లీ మ్యూజిక్ సో ఐ యామ్ హ్యాపీ</text>
<text start="376.72" dur="3.84">కోర్ట్ ఇస్ ఏ గుడ్ ఫిల్మ్ దట్స్ ఇట్ ఫ్రమ్</text>
<text start="379.039" dur="2.801">దిస్ రివ్యూ డు సబ్స్క్రైబ్ టు గలాటా</text>
<text start="380.56" dur="6.079">తెలుగు ఫర్ మోర్ రివ్యూస్ అండ్</text>
<text start="381.84" dur="8.24">ఇంటర్వ్యూస్ డే డే టేస్ట్ డే టేస్ట్ డైలీ</text>
<text start="386.639" dur="6.881">హాలిడేస్ నాలే అది నమ్మ జిటి హాలిడేస్ దా</text>


"""



In [17]:
import re


# Use regular expression to extract text between <text ...> and </text>
matches = re.findall(r'<text[^>]*>(.*?)</text>', input_text)

# Join extracted sentences with space
result = ' '.join(matches)

print(result)


హలో ఆల్ వెల్కమ్ టు తై వ్యూ అండ్ హియర్ ఇస్ అవర్ నాన్ స్పాయిలర్ రివ్యూ ఆఫ్ ది ఫిలిం కోర్ట్ హిట్ త్రీ చూడడానికి రెడీ అయిపోండి భయ్యా మన వర్డ్స్ లో చెప్పాలంటే నచ్చింది సినిమా నాని చెప్పాక డౌట్ ఏంటి భయ్యా అంత కాన్ఫిడెంట్ గా కోర్ట్ సినిమా నచ్చితేనే హిట్ త్రీ చూడండి అని చెప్పారు ఆయన మంచి సినిమా వస్తే చాలు చూడండి అని చెప్పి ఆడియన్స్ ని రిక్వెస్ట్ చేస్తారు ఆయన అసలు నాని అంతలా ఎందుకు రిక్వెస్ట్ చేశారు సినిమాలో ఇంతకు ఏముంది అనేది రివ్యూ లో డిస్కస్ చేసుకుందాం కథ విషయానికి వస్తే చంద్రశేఖర్ అనే అబ్బాయి జాబిలి అనే అమ్మాయిని లవ్ చేస్తాడు కానీ జాబిలి వాళ్ళ ఫ్యామిలీ రిలేటివ్ మంగపతి తన మీద పాక్సో కేస్ పెట్టేస్తాడు ఈ కేసు నుంచి లాయర్ సూర్యతేజ చంద్రశేఖర్ ని కాపాడగలిగాడా లేదా అనేదే స్టోరీ ఫస్ట్ అఫ్ ఆల్ సినిమా నుంచి బయటికి వస్తున్నప్పుడు క్లాప్స్ విన్నా నాకు మామూలుగానే కోర్ట్ రూమ్ డ్రామాస్ అంటే చాలా ఇష్టం ఎంగేజింగ్ గా తీస్తే ఎన్ని కోర్ట్ రూమ్ డ్రామాలైనా చూడొచ్చు అని నమ్ముతాను నేనైతే లవ్ ట్రాక్ చాలా బాగా రాసుకున్నారు అండ్ లవ్ ట్రాక్ కేస్ ని లీడ్ చేయడం త్రిల్లింగ్ గా రాసుకున్నారు అండ్ కోర్ట్ అనే టైటిల్ పెట్టినం